<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Proyectos/04_Proyecto_Transformers_Preentrenados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Proyecto 4: atención y modelos preentrenados</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

**Objetivo:** implementar modelos de **atención** y modelos **preentrenados** para el modelado de texto, evaluarlos con rigor y analizar sus riesgos, aplicando lo aprendido en el módulo 4 (CNN para texto, atención y Transformers, BERT y GPT, fine-tuning, aplicaciones y ética).

**Modalidad:** grupos de 2 a 3 personas.

Elijan **un problema real**, preferiblemente con textos **en español** y de interés para una organización o comunidad colombiana (atención al ciudadano, quejas de usuarios de servicios públicos, noticias regionales, documentos académicos de la universidad, opiniones sobre turismo en Boyacá, mensajes de soporte...). El proyecto tiene tres partes:

**Parte A — Modelo ajustado (obligatoria)**

Una tarea de **clasificación de textos** o de **etiquetado de secuencias** (NER, detección de entidades específicas del dominio) en la que se requiere:

1. Líneas base del curso: TF-IDF con regresión logística y una CNN o una RNN entrenada desde cero.
2. **Fine-tuning** de al menos dos modelos preentrenados con `transformers` (por ejemplo BETO, RoBERTuito, MarIA/RoBERTa-BNE, un modelo multilingüe), con varias semillas.
3. Una **curva de aprendizaje**: desempeño en función del número de ejemplos etiquetados (por ejemplo 100, 500, 2 000 y todos), comparando el mejor modelo preentrenado con la mejor línea base.

**Parte B — Modelos sin ajuste (elige una opción)**

- **B1. Zero-shot y few-shot:** resolver la misma tarea de la Parte A sin fine-tuning, con clasificación *zero-shot* basada en NLI o con un LLM pequeño con instrucciones y ejemplos en el prompt (*few-shot*). Compararlo con la curva de aprendizaje: ¿a cuántos ejemplos etiquetados equivale?
- **B2. Generación aumentada con recuperación (RAG):** un sistema que responda preguntas sobre una colección de documentos del dominio (reglamentos, normas, guías...), recuperando los fragmentos relevantes con embeddings y generando la respuesta con un LLM. Evaluarlo con un conjunto de preguntas escritas por el grupo, incluidas preguntas **sin respuesta** en los documentos.
- **B3. Resumen o traducción:** aplicar un modelo preentrenado de resumen o de traducción a textos del dominio y evaluarlo con métricas automáticas (ROUGE o BLEU) y con una **evaluación humana** de fidelidad (¿el resultado contiene información que no está en el original?).

**Parte C — Auditoría ética (obligatoria)**

Evaluar el mejor sistema con las técnicas del cuaderno 19: pruebas con contrafactuales sobre grupos de personas relevantes para el dominio, análisis de privacidad de los datos y, si se usa un modelo generativo, medición de alucinaciones. Redactar una **tarjeta del modelo** (*model card*) con su uso previsto, sus datos, sus métricas (globales y por subgrupo) y sus limitaciones.

En todos los casos se valorarán la validación sin fugas de información ni contaminación, las métricas adecuadas, la comparación con líneas base y el análisis de errores.

**Recursos de cómputo:** todo debe poder ejecutarse en Google Colab con una GPU T4. Usen modelos de tamaño *base* (≈ 100–350 millones de parámetros) para el fine-tuning y LLM de como máximo 3 000 millones de parámetros. Documenten el tiempo de cada entrenamiento.

**Entregables:**

- Este cuaderno completo, ejecutado de principio a fin sin errores (*Kernel → Restart & Run All*).
- La tarjeta del modelo (al final de este cuaderno o como archivo aparte).
- Una presentación de 10 minutos con los hallazgos principales.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Definición del problema y de los datos</span>](#definicion-del-problema-y-de-los-datos)
- [<span style="color: #005C99;">Parte A: preparación y líneas base</span>](#parte-a-preparacion-y-lineas-base)
- [<span style="color: #005C99;">Parte A: fine-tuning</span>](#parte-a-fine-tuning)
- [<span style="color: #005C99;">Parte A: curva de aprendizaje y evaluación final</span>](#parte-a-curva-de-aprendizaje-y-evaluacion-final)
- [<span style="color: #005C99;">Parte B: modelos sin ajuste</span>](#parte-b-modelos-sin-ajuste)
- [<span style="color: #005C99;">Parte C: auditoría ética</span>](#parte-c-auditoria-etica)
- [<span style="color: #005C99;">Tarjeta del modelo</span>](#tarjeta-del-modelo)
- [<span style="color: #005C99;">Conclusiones</span>](#conclusiones)
- [<span style="color: #005C99;">Rúbrica de evaluación</span>](#rubrica-de-evaluacion)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib seaborn scikit-learn datasets "transformers>=5" accelerate torch

In [ ]:
import os
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import transformers
from transformers import AutoTokenizer, pipeline, set_seed

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
set_seed(SEED)  # fija las semillas de Python, NumPy y PyTorch
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | PyTorch {torch.__version__} | dispositivo: {dispositivo}")

<a id="definicion-del-problema-y-de-los-datos"></a>
<h2 style="color: #007ACC;">Definición del problema y de los datos</h2>

- Describe el problema, quién usaría el sistema y qué decisiones apoyaría. ¿Qué tipo de error es más costoso y para quién?
- Describe las fuentes de datos, su licencia, tamaño, idioma y variedad lingüística. Si los datos se etiquetaron para el proyecto, describe el protocolo y mide el **acuerdo entre anotadores** en una muestra (por ejemplo, con el kappa de Cohen).
- Si hay datos personales, explica cómo se anonimizaron (cuaderno 19).

In [ ]:
# Escribe tu código aquí

<a id="parte-a-preparacion-y-lineas-base"></a>
<h2 style="color: #007ACC;">Parte A: preparación y líneas base</h2>

- Análisis exploratorio: distribución de clases, longitudes en palabras y en tokens del modelo preentrenado, ejemplos.
- División en entrenamiento, validación y prueba, justificada (¿al azar, por fecha, por fuente?). Comprueba que no haya duplicados entre conjuntos.
- Líneas base: TF-IDF + regresión logística y una CNN o RNN desde cero, con sus métricas en validación.

In [ ]:
# Escribe tu código aquí

<a id="parte-a-fine-tuning"></a>
<h2 style="color: #007ACC;">Parte A: fine-tuning</h2>

- Justifica la elección de al menos dos modelos preentrenados (idioma, dominio de preentrenamiento, tamaño, licencia). Lee y resume sus tarjetas.
- Fine-tuning con `Trainer`: hiperparámetros, tiempo de entrenamiento y curvas de validación.
- Repite con al menos 3 semillas y reporta media y desviación estándar.

In [ ]:
# Escribe tu código aquí

<a id="parte-a-curva-de-aprendizaje-y-evaluacion-final"></a>
<h2 style="color: #007ACC;">Parte A: curva de aprendizaje y evaluación final</h2>

- Curva de aprendizaje del mejor modelo preentrenado y de la mejor línea base con distintas cantidades de datos etiquetados.
- Evalúa el modelo elegido **una sola vez** en prueba. Reporta métricas por clase y la matriz de confusión.
- Análisis de errores: lee al menos 20 errores y clasifícalos (error de etiqueta, ambigüedad, falta de contexto, error del modelo...).

In [ ]:
# Escribe tu código aquí

<a id="parte-b-modelos-sin-ajuste"></a>
<h2 style="color: #007ACC;">Parte B: modelos sin ajuste</h2>

- **B1:** prompts o hipótesis probados, resultados en validación y en prueba, comparación con la curva de aprendizaje y costo por predicción.
- **B2:** fragmentación de los documentos, modelo de embeddings, número de fragmentos recuperados, prompt del LLM. Evaluación de la recuperación (¿el fragmento correcto está entre los recuperados?) y de las respuestas (correctas, incorrectas, abstenciones), incluidas las preguntas sin respuesta.
- **B3:** métricas automáticas frente a líneas base (por ejemplo *lead-3*) y evaluación humana de fidelidad en al menos 30 ejemplos, con ejemplos de alucinaciones si las hay.

In [ ]:
# Escribe tu código aquí

<a id="parte-c-auditoria-etica"></a>
<h2 style="color: #007ACC;">Parte C: auditoría ética</h2>

- Pruebas con contrafactuales: plantillas o ejemplos reales en los que solo cambia una referencia a un grupo (género, región, etnia, nacionalidad...). Reporta las diferencias en las predicciones.
- Métricas por subgrupo, si los datos lo permiten (por ejemplo, por región o por longitud del texto).
- Explicaciones locales (oclusión u otro método) de algunos aciertos y errores.
- Si usas un modelo generativo: mide alucinaciones e inyección de instrucciones con ejemplos diseñados por el grupo.

In [ ]:
# Escribe tu código aquí

<a id="tarjeta-del-modelo"></a>
<h2 style="color: #007ACC;">Tarjeta del modelo</h2>

Redacta la tarjeta del modelo siguiendo a Mitchell et al. (2019): detalles del modelo, uso previsto y usos no recomendados, factores relevantes (grupos, variedades lingüísticas), métricas, datos de evaluación y de entrenamiento, consideraciones éticas, advertencias y recomendaciones.

In [ ]:
# (Opcional) código de apoyo para la tarjeta del modelo

<a id="conclusiones"></a>
<h2 style="color: #007ACC;">Conclusiones</h2>

- ¿Cuánto aportó el preentrenamiento frente a las líneas base? ¿A partir de cuántos ejemplos etiquetados dejó de valer la pena?
- ¿Qué opción de la Parte B recomendarían a una organización con pocos recursos de cómputo y de etiquetado?
- ¿Qué riesgos encontraron en la auditoría y cómo los mitigarían antes de un despliegue real?

In [ ]:
# Escribe tu código aquí

<a id="rubrica-de-evaluacion"></a>
<h2 style="color: #007ACC;">Rúbrica de evaluación</h2>

| Criterio | Peso | Se espera |
|----------|------|-----------|
| Problema, datos y preparación | 10 % | Problema relevante; datos bien documentados; divisiones sin fugas ni duplicados |
| Parte A: líneas base y fine-tuning | 25 % | Comparación justa; varias semillas; hiperparámetros y tiempos documentados |
| Parte A: curva de aprendizaje y evaluación | 15 % | Curva bien construida e interpretada; prueba usada una sola vez; análisis de errores |
| Parte B: modelos sin ajuste | 20 % | Implementación correcta; evaluación con casos difíciles (sin respuesta, alucinaciones) |
| Parte C: auditoría ética y tarjeta del modelo | 20 % | Pruebas concretas con resultados medidos; tarjeta completa y honesta |
| Calidad del cuaderno y presentación | 10 % | Código limpio y reproducible; narrativa clara; conclusiones basadas en los resultados |